# 27 · Pooled monotone model, YouTube drift, copy-paste campaigns, posts without reactions (E61-E64)

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 220, "display.max_columns", 30)
import eda_utils as eu, extra_models as em, history as H, more_models as mm

In [2]:
f, meta = em.load(ROOT)

## E61 · One pooled model with monotone constraints

In [3]:
P, T = mm.pooled(f, meta); SP = H.summarise(P); ST = H.summarise(T)
pd.concat([SP, ST]).pivot(index="model", columns="platform", values="MAE")[mm.PLATS].round(3)

platform,tiktok,youtube,twitter
model,,,
LOPO CatBoost,0.412,0.632,0.706
"LOPO CatBoost, monotone",0.425,0.615,0.722
LOPO ridge on likes + comments (E13),0.313,0.611,0.650
per platform (final model),0.271,0.453,0.467
"per platform, monotone",0.274,0.479,0.478
pooled with platform column,0.272,0.465,0.471
"pooled, monotone",0.277,0.496,0.484


In [4]:
G61 = pd.concat([pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P, p, "per platform (final model)", k)))} for p in P.platform.unique() for k in ["per platform, monotone", "pooled with platform column", "pooled, monotone"]]), pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(T, p, "LOPO ridge on likes + comments (E13)", k)))} for p in T.platform.unique() for k in ["LOPO CatBoost", "LOPO CatBoost, monotone"]])]); G61.round(3)

,platform,model,gain,CI low,CI high
0,twitter,"per platform, monotone",-0.011,-0.018,-0.005
1,twitter,pooled with platform column,-0.004,-0.008,0.000
2,twitter,"pooled, monotone",-0.017,-0.025,-0.010
3,youtube,"per platform, monotone",-0.026,-0.033,-0.019
4,youtube,pooled with platform column,-0.012,-0.017,-0.007
5,youtube,"pooled, monotone",-0.043,-0.055,-0.032
6,tiktok,"per platform, monotone",-0.004,-0.006,-0.001
7,tiktok,pooled with platform column,-0.001,-0.004,0.002
8,tiktok,"pooled, monotone",-0.007,-0.010,-0.003
0,twitter,LOPO CatBoost,-0.056,-0.075,-0.037


## E62 · YouTube views per like by month

In [5]:
D = mm.youtube_drift(ROOT); D.pivot_table(index=["platform", "month"], columns="model", values=["median_age_days", "median_views_per_like", "bias"], sort=False).round(3)

median_age_days                         median_views_per_like                                bias                        
model                final model final model + check age           final model final model + check age final model final model + check age
platform month                                                                                                                            
tiktok   2026-04          93.077                  93.077                 1.371                   1.371      -0.048                  -0.021
         2026-05          70.494                  70.494                 1.397                   1.397      -0.024                  -0.010
         2026-06          41.671                  41.671                 1.391                   1.391      -0.025                  -0.017
         2026-07          14.279                  14.279                 1.372                   1.372      -0.004                  -0.013
         2026-08           7.740                   7.740                 1.335                   1.335      -0.009                  -0.046
youtube  2026-04         100.348                 100.348                 1.557                   1.557      -0.108                  -0.062
         2026-05          78.227                  78.227                 1.568                   1.568      -0.037                  -0.003
         2026-06          45.382                  45.382                 1.540                   1.540      -0.038                  -0.019
         2026-07          17.261                  17.261                 1.301                   1.301      -0.038                  -0.026
         2026-08           0.947                   0.947                 1.126                   1.126       0.006                  -0.057
twitter  2026-04          89.107                  89.107                 1.230                   1.230      -0.018                  -0.031
         2026-05          72.004                  72.004                 1.146                   1.146       0.031                  -0.014
         2026-06          37.384                  37.384                 1.342                   1.342      -0.067                  -0.030
         2026-07           7.727                   7.727                 1.301                   1.301      -0.094                  -0.066
         2026-08           0.565                   0.565                 1.230                   1.230      -0.055                  -0.091

## E63 · Copy-paste campaigns

In [6]:
Ct, top, dc = mm.copies(ROOT); Ct.round(3)

,platform,is_copy,posts,median_views,median_log_views_per_like,median_reactions,median_followers,reach_share
0,facebook,False,2341,NaN,NaN,3.0,59252.0,0.925
1,facebook,True,159,NaN,NaN,1.0,68227.5,0.075
2,instagram,False,2355,NaN,NaN,1.0,7694.0,0.999
3,instagram,True,145,NaN,NaN,0.0,NaN,0.001
4,tiktok,False,2470,728.5,1.377,32.0,1427.0,0.992
5,tiktok,True,30,291.0,1.618,7.0,2559.0,0.008
6,twitter,False,2404,26.0,1.255,0.0,385.0,0.982
7,twitter,True,96,24.5,1.389,0.0,206.0,0.018
8,youtube,False,2467,408.5,1.376,7.0,27800.0,0.880
9,youtube,True,33,542.0,1.639,0.0,51100.0,0.120


In [7]:
top

,platform,brands with copies,brands with > 50% reach from copies
0,facebook,18,0
1,instagram,10,0
2,tiktok,14,2
3,twitter,29,7
4,youtube,15,6


In [8]:
CE = mm.copy_errors(f, meta, dc); CE.round(3)

,platform,posts,n,MAE,bias
0,tiktok,copied text,29,0.577,-0.053
1,tiktok,unique text,2462,0.270,-0.019
2,youtube,copied text,33,0.742,-0.259
3,youtube,unique text,2408,0.452,-0.036
4,twitter,copied text,66,0.548,-0.137
5,twitter,unique text,1737,0.463,-0.036


## E64 · Posts without likes and comments

In [9]:
Z = mm.zero_reactions(ROOT, f, meta, ROOT / "data" / "processed" / "embeddings_e5_cache.pkl"); SZ = H.summarise(Z); SZ.round(3)

,platform,model,posts,MAE,CI low,CI high,bias
0,youtube,final model (all posts),665,0.662,0.616,0.718,-0.017
1,youtube,"all posts, + text and context",665,0.667,0.621,0.724,-0.021
2,youtube,"zero-reaction posts only, text and context",665,0.672,0.627,0.728,-0.045
3,twitter,final model (all posts),991,0.493,0.466,0.520,-0.015
4,twitter,"all posts, + text and context",991,0.462,0.438,0.486,0.025
5,twitter,"zero-reaction posts only, text and context",991,0.482,0.457,0.507,-0.030


In [10]:
G64 = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(Z, p, "final model (all posts)", k)))} for p in Z.platform.unique() for k in ["all posts, + text and context", "zero-reaction posts only, text and context"]]); G64.round(3)

,platform,model,gain,CI low,CI high
0,youtube,"all posts, + text and context",-0.005,-0.029,0.020
1,youtube,"zero-reaction posts only, text and context",-0.010,-0.041,0.020
2,twitter,"all posts, + text and context",0.030,0.016,0.045
3,twitter,"zero-reaction posts only, text and context",0.011,-0.003,0.025


## Takeaways
See `docs/en/experiments.md` (E61-E64). Numbers: `data/processed/more_models_numbers.json`.

In [11]:
json.dump({"pooled": SP.to_dict("records"), "lopo": ST.to_dict("records"), "gains_61": G61.to_dict("records"), "youtube_drift": D.to_dict("records"), "copies": Ct.to_dict("records"), "copy_brands": top.to_dict("records"), "copy_errors": CE.to_dict("records"), "zero_reactions": SZ.to_dict("records"), "gains_64": G64.to_dict("records")},
          open(ROOT / "data" / "processed" / "more_models_numbers.json", "w"), indent=1, default=float)